In [1]:
import shutil
import os
# unziped  YOLO_Bounding_box_labelling 

shutil.unpack_archive("YOLO_Bounding_box_labelling.zip", "YOLO_Bounding_box_labelling")

print(len(os.listdir("YOLO_Bounding_box_labelling")))

1092


In [2]:
!pip install ultralytics

In [3]:
import os
import shutil
image_dir = "tiles_jpg_final_unzipped"
label_dir = "YOLO_Bounding_box_labelling"
base = "/home3/zpdv81/YOLO_Bounding_box_detection"


# create YOLO folder structure
for split in ["train","val"]:
    os.makedirs(os.path.join(base,"images",split),exist_ok = True)
    os.makedirs(os.path.join(base,"labels",split),exist_ok = True)
    
# width of original raster
raster_width = 20000
val_cutoff = raster_width*0.8 # 80% train 20% validation

# assign a tile to train or validation based on its left x_coordinate

def get_split(tile_name):
    parts = tile_name.split("_")
    left = int(parts[2])
    if left < val_cutoff:
        return "train"
    else:
        return "val"
    
train_count = 0
val_count = 0

# copy each image and its yolo label file
for label_file in os.listdir(label_dir):
    if not label_file.endswith(".txt"):      # only process yolo text label files
        continue
        
    # remove .txt to get matching image file name
    tile_name = os.path.splitext(label_file)[0]
    image_path = os.path.join(image_dir , tile_name + ".jpg")
    
    
    # remove .txt to get matching image file name
    if not os.path.exists(image_path):
        print(f"Missing image for : {label_file}")
        continue
        
    # check whether this tile belongs in train or validation
    split = get_split(tile_name)
    
    # source paths
    label_path = os.path.join(label_dir,label_file)
    
    # path destination 
    output_label_path = os.path.join(base,'labels',split,label_file)
    output_image_path = os.path.join(base,'images',split ,tile_name + ".jpg")
    
    # copy 
    shutil.copy(label_path , output_label_path)
    shutil.copy(image_path , output_image_path)
    
    if split == "train":
        train_count +=1
    else:
        val_count +=1
        
print(f" Train images : {train_count}")

print(f" Validation images : {val_count}")


total = train_count + val_count
print(f" Train percentage : {train_count/total * 100:.2f}%")
print(f" validation percentage : {val_count / total * 100 :.2f}%")

# create YOLO data.yaml file
data_yaml_content = f"""train: {base}/images/train
val: {base}/images/val

nc: 1

names:
  0: ridge_and_furrow


"""

yaml_path = os.path.join(base,"data.yaml")

with open(yaml_path,"w")as file:
    file.write(data_yaml_content)
    
print(f"data.yaml saved to : {yaml_path}")

 Train images : 955
 Validation images : 137
 Train percentage : 87.45%
 validation percentage : 12.55%
data.yaml saved to : /home3/zpdv81/YOLO_Bounding_box_detection/data.yaml


In [4]:
import sys
print(sys.executable)

/apps/jupyterhub/jupyterenv/bin/python3


In [5]:
import sys
!{sys.executable} -m pip install --target=/home3/zpdv81/pylibs ultralytics

  Using cached ultralytics-8.4.123-py3-none-any.whl (1.4 MB)
  Using cached opencv_python-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl (73.8 MB)
  Using cached matplotlib-3.7.5-cp38-cp38-manylinux_2_12_x86_64.manylinux2010_x86_64.whl (9.2 MB)
  Using cached torchvision-0.19.1-cp38-cp38-manylinux1_x86_64.whl (7.0 MB)
  Using cached PyYAML-6.0.3-cp38-cp38-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (806 kB)
  Using cached pillow-10.4.0-cp38-cp38-manylinux_2_28_x86_64.whl (4.5 MB)
  Using cached torch-2.4.1-cp38-cp38-manylinux1_x86_64.whl (797.1 MB)
  Using cached psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl (155 kB)
  Using cached numpy-1.24.4-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
  Using cached nvidia_ml_py-13.610.43-py3-none-any.whl (53 kB)
  Using cached filelock-3.16.1-py3-none-any.whl (16 kB)
  Using cached polars-1.8.2-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (32.5 M

In [6]:
sys.path.insert(0, "/home3/zpdv81/pylibs")

from ultralytics import YOLO

In [8]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{base}/data.yaml",
    epochs=10,       # sanity check first
    imgsz=640,
    batch=16,
    device=0,
    workers = 2,
    project="/home3/zpdv81/YOLO_BOUNDING_BOX_DETECTION",
    name="sanity_check",
)

New https://pypi.org/project/ultralytics/8.4.123 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.104 🚀 Python-3.8.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA A100 80GB PCIe MIG 1g.10gb, 9728MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home3/zpdv81/YOLO_Bounding_box_detection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=30

/home3/zpdv81/pylibs/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to /home3/zpdv81/YOLO_BOUNDING_BOX_DETECTION/sanity_check-6/labels.jpg... 
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /home3/zpdv81/YOLO_BOUNDING_BOX_DETECTION/sanity_check-6
Starting training for 10 epochs...
Closing dataloader mosaic

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/10      2.05G      1.698      3.253      2.072         16        640: 100% ━━━━━━━━━━━━ 60/60 3.4it/s 17.7s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.4it/s 2.1s0.5s
                   all        137        175      0.224      0.114      0.118     0.0451

      Epoch    GP

In [10]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{base}/data.yaml",
    epochs=100,      
    imgsz=640,
    batch=16,
    seed = 42,
    device=0,
    workers = 2,
    project="/home3/zpdv81/YOLO_BOUNDING_BOX_DETECTION",
    name="YOLO_BOUNDING_BOX_MODEL_1",
)

New https://pypi.org/project/ultralytics/8.4.123 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.104 🚀 Python-3.8.10 torch-2.4.1+cu121 CUDA:0 (NVIDIA A100 80GB PCIe MIG 1g.10gb, 9728MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home3/zpdv81/YOLO_Bounding_box_detection/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=3

/home3/zpdv81/pylibs/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to /home3/zpdv81/YOLO_BOUNDING_BOX_DETECTION/YOLO_BOUNDING_BOX_MODEL_1/labels.jpg... 
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /home3/zpdv81/YOLO_BOUNDING_BOX_DETECTION/YOLO_BOUNDING_BOX_MODEL_1
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      3.69G      1.384      2.559      1.767         18        640: 100% ━━━━━━━━━━━━ 60/60 3.5it/s 17.0s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.7it/s 1.9s0.6s
                   all        137        175      0.174       0.04      0.104     0.0358

      Epoch    GPU_m